# 7.2. Convolutions for Images

Let's build a simple, minimal example of a convolutional layer to understand what it does and why we might need it for non-trivial image classification tasks involving many large, colored, high-resolution images. As usual, the software we need is listed below.

1. Python 3.12
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
!cat requirements.txt

absl-py==2.4.0
attrs==26.1.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.7
jupyterlab-git==0.53.0
jupyter-resource-usage==1.2.1
mindspore==2.8.0
ml-dtypes==0.5.4
sympy==1.14.0
tornado==6.5.5


In [2]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 7.2.1. The Cross-Correlation Operation

Consider the following matrices.

1. Input "image": $\mathbf{X} = \bigl( \begin{smallmatrix} 0 & 1 & 2 \\ 3 & 4 & 5 \\ 6 & 7 & 8 \end{smallmatrix} \bigr)$
1. Convolutional kernel: $\mathbf{K} = \bigl( \begin{smallmatrix} 0 & 1 \\ 2 & 3 \end{smallmatrix} \bigr)$

We can obtain the output $\mathbf{Y} = \mathbf{X} \ast \mathbf{K} = \bigl( \begin{smallmatrix} 0 \times 0 + 1 \times 1 + 3 \times 2 + 4 \times 3 & 1 \times 0 + 2 \times 1 + 4 \times 2 + 5 \times 3 \\ 3 \times 0 + 4 \times 1 + 6 \times 2 + 7 \times 3 & 4 \times 0 + 5 \times 1 + 7 \times 2 + 8 \times 3 \end{smallmatrix} \bigr) = \bigl( \begin{smallmatrix} 19 & 25 \\ 37 & 43 \end{smallmatrix} \bigr)$ by shifting the kernel across our "image" and summing the elementwise products of both matrices. This is known as _cross-correlation_.

Notice the resulting matrix is smaller than our input "image". In general, given an image $\mathbf{X}_{n_h \times n_w}$ and kernel $\mathbf{K}_{k_h \times k_w}$, the resulting matrix $\mathbf{Y}$ has shape $(n_h - k_h + 1) \times (n_w - k_w + 1)$.

Let's see an implementation of the above with MindSpore.

In [4]:
import mindspore.ops as ops
from mindspore import dtype as mstype

def corr2d(X, K):
    """Compute 2D cross-correlation."""
    h, w = K.shape
    Y = ops.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1), dtype=mstype.int64)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = ops.sum(X[i:i+h, j:j+w] * K)
    return Y

X = ops.reshape(ops.arange(9), (3, 3))
K = ops.reshape(ops.arange(4), (2, 2))
Y = corr2d(X, K)
X, K, Y

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
2026-05-02 11:12:21.173187: E external/org_tensorflow/tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute is_closed which is not in the op definition: Op<name=Range; signature=start:Tidx, limit:Tidx, delta:Tidx -> output:Tidx; attr=Tidx:type,default=DT_INT32,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_INT8, DT_INT16, DT_INT32, DT_INT64, DT_UINT16, DT_UINT32]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Range1}}


(Tensor(shape=[3, 3], dtype=Int64, value=
 [[0, 1, 2],
  [3, 4, 5],
  [6, 7, 8]]),
 Tensor(shape=[2, 2], dtype=Int64, value=
 [[0, 1],
  [2, 3]]),
 Tensor(shape=[2, 2], dtype=Int64, value=
 [[19, 25],
  [37, 43]]))

## 7.2.2. Convolutional Layers

We are now ready to implement our first convolutional layer. Define the kernel matrix $\mathbf{K}$ and scalar bias $b$. Our convolutional layer computes the output matrix $\mathbf{Y}$ defined below.

$$
\begin{align}
\mathbf{Y} = \mathbf{X} \ast \mathbf{K} + b
\end{align}
$$

Notice how similar it is to our fully connected layer, except the weight vector $\mathbf{w}$ is replaced with the kernel matrix $\mathbf{K}$ and matrix-vector multiplication is replaced with the cross-correlation operation.

As usual, we default to initializing our kernel $\mathbf{K}$ with random weights from the standard normal distribution and the scalar bias $b$ to zero. [`mindspore.nn.Conv2d`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/nn/mindspore.nn.Conv2d.html) is already defined by MindSpore so let's call our convolutional layer `MyConv2d`.

In [5]:
import mindspore.nn as nn

class MyConv2d(nn.Cell):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = mindspore.Parameter(ops.randn(*kernel_size))
        self.bias = mindspore.Parameter(ops.zeros(1))

    def construct(self, X):
        y_hat = corr2d(X, self.weight) + self.bias
        return y_hat

my_conv2d_layer = MyConv2d(kernel_size=(1, 2))
my_conv2d_layer

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: Sy

MyConv2d()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: Sy

## 7.2.3. Object Edge Detection in Images

TODO